# Question Answering with BERT and SQuAD Spanish (using Hugging Face)

Author: Vladimir Araujo

Based on: https://www.spark64.com/post/machine-comprehension



## Instrucciones Generales

El siguiente práctico se realiza individualmente. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondida en celdas de texto. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre:** Roberto Araneda

El siguiente práctico cuanta con 2 secciones donde cada una contendrá 1 o más actividades a realizar. Algunas actividades correspondrán a escribir código y otras a responder preguntas.

**Importante.** Para facilitar su ejecución, cada sección puede ser ejecutada independientemente.

Se recomienda **fuertemente** revisar las secciones donde se entrega código porque algunas actividades de código pueden reutilizar el mismo código pero con cambios en algunas líneas.

## 1.0 Introduction

Question Answering (QA) is a challenging task that NLP tries to solve. The aim is to provide solution to queries expressed in natural language automatically (Hovy, Gerber, Hermjakob, Junk, and Lin 2000). For instance, given the following context:

> Quito, oficialmente San Francisco de Quito, es la capital de la República del Ecuador, de la Provincia de Pichincha y la capital más antigua de Sudamérica. Es la ciudad más poblada del Ecuador,​ con 2 millones de habitantes en el área urbana, y aproximadamente 3 millones en todo el Área metropolitana.

We ask the question

> ¿Cuál es la población de Quito?

We expect the QA system responds with something like this:

> 2 millones

Since 2017, transformer models have been shown to outperform existing approaches for this task. Currently, many pretrained transformer models exist, including BERT, GPT-2, XLNet.

This tutorial shows how you can fine-tune BERT for the task of QA and use it for inference. We will use the transformer library built by [Hugging Face](https://huggingface.co/), which is an extremely useful implementation of the transformer models in both TensorFlow and PyTorch. You can just use a fine-tuned model from their [model hub](https://huggingface.co/models).

This tutorial is for educational purposes with which we will learn to finetune a BERT model and use it with your own data.

## Using BERT-based model for QA

<figure>
<center>
<img src='https://miro.medium.com/max/1840/1*QhIXsDBEnANLXMA0yONxxA.png' width="500" />
</center>
</figure>

*   Input is the $Question$ tokens and the $Paragraph$ tokens separated by the special token $[SEP]$.
*   The final hidden vector of BERT is $T_i$
*   New parameters learned during fine-tuning are a start vector $S$ and an end vector $E$.
*   The probability of word $i$ being the start/end of the answer span is computed as a dot product between $T_{i}$ and $S$ or $E$ followed by a softmax.

## 2.0 Setup

First, we clone and install the Hugging Face transformer library from Github.

In [ ]:
!mkdir -p downloads \
&& cd downloads \
&& git clone --branch v4.52.4 --depth 1 https://github.com/huggingface/transformers.git \
&& cd transformers \
&& pip install -e .


In [ ]:
!pip install evaluate==0.4.4
!pip install accelerate==1.8.0 -U
!pip install datasets==3.6.0 -U

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

## 3.0 Train Model

This is where we can train our own model.

### 3.1 Get Training and Evaluation Data

The [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/) is a reading comprehension dataset, consisting of questions posed by crowdworkers on a set of Wikipedia articles, where the answer to every question is a segment of text, or span, from the corresponding reading passage, or the question might be unanswerable. In this tutorial we will use a Spanish version of this dataset.

Read more about this dataset here: https://github.com/ccasimiro88/TranslateAlignRetrieve

Now get the Spanish SQuAD V2.0 dataset. We use `datasets` library to load the data.



In [ ]:
from datasets import load_dataset

dataset = load_dataset("TheTung/squad_es_v2", "small", trust_remote_code=True)

In [ ]:
dataset

### 3.2 Dataset Exploration

Let's explore an example of the dataset. You need to change `id` variable if you want to change the example.

In [ ]:
id = 7

In [ ]:
dataset['validation'][id]['context']

In [ ]:
dataset['validation'][id]['question']

In [ ]:
dataset['validation'][id]['answers']

### 3.3 Run training (Optional)

We can now train the model with the training set.

**Notes about parameters:**

`per_gpu_train_batch_size` specifies the number of training examples per iteration per GPU.

`save_steps` specifies number of steps before it outputs a checkpoint file. I've increased it to save disk space.

`num_train_epochs` sets the number of epochs, two epochs are recommended. It's currently set to one for the purpose of time.

`version_2_with_negative` is required for SQuAD V2.0. If training with V1.1, take out this flag.

NOTE: it takes about 1 hour to train an epoch! If you don't want to wait this long, feel free to skip this step and use a pretrained model!

In [ ]:
!python downloads/transformers/examples/pytorch/question-answering/run_qa.py \
  --model_name_or_path dccuchile/bert-base-spanish-wwm-cased \
  --dataset_name TheTung/squad_es_v2 \
  --dataset_config_name small \
  --do_train \
  --do_eval \
  --per_device_train_batch_size 12 \
  --learning_rate 3e-5 \
  --num_train_epochs 2 \
  --max_seq_length 384 \
  --doc_stride 128 \
  --version_2_with_negative \
  --output_dir /content/model_output

In [ ]:
!ls -lah /content/model_output 2>/dev/null && echo "✅ SIGUE VIVO" || echo "❌ runtime reseteado, /content vacío"

## 4.0 Setup prediction code

Now we can use the Hugging Face library to make predictions using our model. Note that a lot of the code is pulled from `run_squad.py` in the Hugging Face repository, with all the training parts removed.


In [ ]:
# READER NOTE: If an error occurs, please restart sesion and try again.
import os
import torch
import time
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

from transformers import (
    BertTokenizer,
    BertForQuestionAnswering,
    squad_convert_examples_to_features
)

from transformers.data.processors.squad import SquadResult, SquadV2Processor, SquadExample
from transformers.data.metrics.squad_metrics import compute_predictions_logits

os.environ["WANDB_DISABLED"] = "true"

If you have trained your own mode, you need to change the flag `use_own_model` to `True`. However, in the case that you want to use a pre-trained model of the hub, you need to change the flag `use_own_model` to `False`, and define the model variable `model_name_or_path`.

In this tutorial, we will use a pre-trained model on SQuAD spanish called [`mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es`](https://huggingface.co/mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es).

In [ ]:
# READER NOTE: Set this flag to use own model, or use pretrained model in the Hugging Face repository
use_own_model = False

if use_own_model:
  model_name_or_path = "/content/model_output"
else:
  model_name_or_path = "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"

output_dir = ""

# Config
n_best_size = 1
max_answer_length = 30
do_lower_case = True
null_score_diff_threshold = 0.0

def to_list(tensor):
    return tensor.detach().cpu().tolist()

# Setup model
model_class, tokenizer_class = (BertForQuestionAnswering, BertTokenizer)
tokenizer = tokenizer_class.from_pretrained(
    model_name_or_path, do_lower_case=True)
model = model_class.from_pretrained(model_name_or_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

processor = SquadV2Processor()

Esta es una función prestada de `run_squad.py`.  This modified code allows to run predictions we pass in directly as strings, rather .json format like the training/test set.

In [ ]:
def run_prediction(question_texts, context_text):
    """Setup function to compute predictions"""
    examples = []

    for i, question_text in enumerate(question_texts):
        example = SquadExample(
            qas_id=str(i),
            question_text=question_text,
            context_text=context_text,
            answer_text=None,
            start_position_character=None,
            title="Predict",
            is_impossible=False,
            answers=None,
        )

        examples.append(example)

    features, dataset = squad_convert_examples_to_features(
        examples=examples,
        tokenizer=tokenizer,
        max_seq_length=384,
        doc_stride=128,
        max_query_length=64,
        is_training=False,
        return_dataset="pt",
        threads=1,
    )

    eval_sampler = SequentialSampler(dataset)
    eval_dataloader = DataLoader(dataset, sampler=eval_sampler, batch_size=10)

    all_results = []

    for batch in eval_dataloader:
        model.eval()
        batch = tuple(t.to(device) for t in batch)

        with torch.no_grad():
            inputs = {
                "input_ids": batch[0],
                "attention_mask": batch[1],
                "token_type_ids": batch[2],
            }

            example_indices = batch[3]

            outputs = model(**inputs, return_dict=False)

            for i, example_index in enumerate(example_indices):
                eval_feature = features[example_index.item()]
                unique_id = int(eval_feature.unique_id)

                output = [to_list(output[i]) for output in outputs]

                start_logits, end_logits = output
                result = SquadResult(unique_id, start_logits, end_logits)
                all_results.append(result)

    output_prediction_file = "predictions.json"
    output_nbest_file = "nbest_predictions.json"
    output_null_log_odds_file = "null_predictions.json"

    predictions = compute_predictions_logits(
        examples,
        features,
        all_results,
        n_best_size,
        max_answer_length,
        do_lower_case,
        output_prediction_file,
        output_nbest_file,
        output_null_log_odds_file,
        False,  # verbose_logging
        True,  # version_2_with_negative
        null_score_diff_threshold,
        tokenizer,
    )

    return predictions

## 5.0 Run predictions

Now for the fun part... testing out your model on different inputs. Pretty rudimentary example here. But the possibilities are endless with this function.

In [ ]:
context = "Quito, oficialmente San Francisco de Quito, es la capital de la República del Ecuador, de la Provincia de Pichincha y la capital más antigua de Sudamérica. Es la ciudad más poblada del Ecuador,​ con 2 millones de habitantes en el área urbana, y aproximadamente 3 millones en todo el Área metropolitana."

questions = ["¿Cuál es la población de Quito?",
             "¿En qué provincia esta ubicado Quito?",
             "¿Cuál es la cápital más antigua de Sudamérica?",
             "¿Qué tan buena es la comida en Ecuador?"]

# Run method
predictions = run_prediction(questions, context)

# Print results
print("Results:")
for i, key in enumerate(predictions.keys()):
  print(questions[i],predictions[key])

## 6.0 Activity

Now is your turn. Use the code in Section 5.0 (previous section) to generate your own predictions. To do that, you must change the context variables and questions. (3 pts)

In [ ]:
# Your code here
context = "El estándar HL7 FHIR define recursos para representar información clínica. El recurso Patient almacena datos demográficos del paciente, mientras que el recurso Observation registra mediciones como signos vitales o resultados de laboratorio. FHIR fue publicado por primera vez en 2014 por la organización Health Level Seven International."

questions = ["¿Qué recurso almacena los datos demográficos del paciente?",   # → Patient
             "¿En qué año se publicó FHIR por primera vez?",                  # → 2014
             "¿Qué organización publicó FHIR?",                               # → Health Level Seven International
             "¿Quién publicó FHIR?",                                          # → Health Level Seven International
             "¿Cuál es el precio de una licencia de FHIR?"]                   # → empty (trampa)

predictions = run_prediction(questions, context)
print("Results:")
for i, key in enumerate(predictions.keys()):
    print(questions[i], predictions[key])


## Análisis del experimento: abstención y sensibilidad a la formulación

Para mi predicción usé un contexto de dominio técnico (estándar HL7 FHIR), distinto
al dominio enciclopédico del entrenamiento (SQuAD-es es Wikipedia traducida). Probé
preguntas respondibles, una pregunta-trampa sin respuesta, y luego experimenté con
el umbral de abstención y con la reformulación de una pregunta.

### Resultados observados

| Pregunta | Respuesta del modelo | Esperado | Veredicto |
|----------|---------------------|----------|-----------|
| ¿Qué recurso almacena los datos demográficos del paciente? | `Patient` | Patient | ✅ Correcto |
| ¿En qué año se publicó FHIR por primera vez? | `2014` | 2014 | ✅ Correcto |
| ¿Qué organización publicó FHIR? | `empty` | Health Level Seven International | ❌ **Falso negativo** |
| ¿Cuál es el precio de una licencia de FHIR? | `empty` | (sin respuesta) | ✅ Abstención correcta |
| ¿Quién publicó FHIR? *(reformulada)* | `organización Health Level Seven International` | Health Level Seven International | ⚠️ Correcto, span impreciso |

### Hallazgos

**1. Falso negativo de abstención en dominio fuera de distribución.**
La pregunta *"¿Qué organización publicó FHIR?"* es respondible (la respuesta está
literal en el contexto), pero el modelo devolvió `empty`. La respuesta es un nombre
propio técnico ("Health Level Seven International") que el modelo no vio durante el
fine-tuning, por lo que generó una distribución start/end poco confiable y el span
nulo (`[CLS]`) ganó.

**2. El ajuste del umbral NO recuperó la respuesta.**
Modifiqué `null_score_diff_threshold` y la pregunta original siguió devolviendo
`empty`. Esto indica que el problema no fue una decisión marginal de calibración
de confianza: el head de span prediction simplemente no produjo un span confiable
para esa formulación de la pregunta.

**3. La reformulación SÍ recuperó la respuesta.**
Cambiar *"¿Qué organización...?"* por *"¿Quién publicó FHIR?"* devolvió la respuesta
correcta. El interrogativo "¿Quién?" se alinea con el rol de agente de la oración
("publicado... **por** la organización..."), mientras que la pregunta de constituyente
"¿Qué organización?" —donde "organización" aparece tanto en la pregunta como pegada
a la respuesta en el contexto— confundió la localización del span.

**4. El span recuperado quedó con bordes imprecisos.**
La respuesta fue `"organización Health Level Seven International"`, incluyendo el
sustantivo común "organización" que no es parte del nombre. Bajo la métrica Exact
Match esto puntúa 0 pese a ser semánticamente correcto; el F1 lo penaliza solo
parcialmente.

### Conclusión

El experimento muestra que, en este modelo extractivo:

- La **abstención de SQuAD v2.0** no siempre significa *"la evidencia no está en el
  texto"*; a veces significa *"no logré mapear esta pregunta a un span"*. El `empty`
  de la pregunta 3 fue un fallo de localización, no un razonamiento de ausencia.
- El rendimiento es **frágil ante el dominio** (vocabulario técnico fuera de
  distribución) y ante la **estructura de la pregunta** (misma información, distinta
  formulación → distinto resultado).
- Las métricas basadas en coincidencia exacta de span (EM/F1) pueden subestimar
  respuestas semánticamente correctas con bordes imperfectos.


---

Based on this tutorial and the class, set whether the following statements are `True` or `False`.


In [ ]:
#@title The SQuAD is a reading comprehension dataset (1 pt)
answer = True #@param ["None","False", "True"] {type:"raw"}

In [ ]:
#@title The BERT model is trained from scratch for the QA task (1 pt)
answer = False #@param ["None","False", "True"] {type:"raw"}

In [ ]:
#@title This model generates the answer word by word (generative approach) (1 pt)
answer = False #@param ["None","False", "True"] {type:"raw"}

## 6.0 Bonus: Attention Vizualization

The attention heads of the Transformer capture the relationships between the tokens. We can explore them to understand which tokens contribute the most to the prediction.

*BertViz* is a tool for visualizing attention in the Transformer model, supporting all models from the HuggingFace library. First, we clone and install the library from Github.

In [ ]:
!pip install bertviz

In [ ]:
def show_head_view(model, tokenizer, sentence_a, sentence_b=None, layer=None, heads=None):
    inputs = tokenizer.encode_plus(sentence_a, sentence_b, return_tensors='pt', add_special_tokens=True)
    input_ids = inputs['input_ids']
    if sentence_b:
        token_type_ids = inputs['token_type_ids']
        attention = model(input_ids, token_type_ids=token_type_ids)[-1]
        sentence_b_start = token_type_ids[0].tolist().index(1)
    else:
        attention = model(input_ids)[-1]
        sentence_b_start = None
    input_id_list = input_ids[0].tolist() # Batch index 0
    tokens = tokenizer.convert_ids_to_tokens(input_id_list)
    head_view(attention, tokens, sentence_b_start, layer=layer, heads=heads)

Now it is necessary to load the pre-trained model on SQuAD.

In [ ]:
from bertviz import head_view
from transformers import BertTokenizer, BertModel

In [ ]:
do_lower_case = True
model = BertModel.from_pretrained(model_name_or_path, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(model_name_or_path, do_lower_case=do_lower_case)

In order to visualize the attentions we need to define `sentence_a` and
`sentence_b`. Remember that the input for BERT QA is `[question,context]`.

Note: You need to set correctly `sentence_b_start` parameter in the `head_view` function depending on the length of your question.

In [ ]:
# Nota: El notebook explota con secuencias de texto muy largas.
sentence_b = "Quito es la capital de la República del Ecuador, de la Provincia de Pichincha."
sentence_a = "¿En qué provincia esta ubicado Quito?"

show_head_view(model, tokenizer, sentence_a, sentence_b)